In [248]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

min_max_scaler = MinMaxScaler()
standard_scaler = StandardScaler()

Prompt:
The source file provided contains security level data of factor elements. The goal is to use random forest to identify the most important features per stock. Once features are identified classify the stock as Value, Growth, or both (assuming most factors are relatively of equal importance). Value factors are as follows: hist_pb, hist_earn_yld, fwd_dps, fwd_dvy. Growth factors are as follows: hist_eps, hist_sps, fwd_pe, fwd_eps, fwd_eps_STg, and fwd_eps_LTg. Group the stocks based on the classification (value, growth, both) per date period and calculate a weighting scheme on each stock in the group based on the magnitude of the combined important feature elements of each stock. Set the target variable to the next monthly stock return. derived the return using the price column.


preprocess null and na values

In [190]:
df = pd.read_csv('test_source.csv')
df['DATE'] = pd.to_datetime(df['DATE'])
#df.set_index('DATE', inplace=True)

recon_dt = list(df['DATE'].drop_duplicates())

dct_src = {}
for date in recon_dt:
    dct_src[date] = df[df['DATE']== date].set_index('DATE')
    
for k,v in dct_src.items():
    #blank data treatment - replace with column set minimum
    #display(v)
    v.dropna(thresh=len(v.columns[2:]), inplace=True) #some stocks listed but have no price
    #print(len(v.columns[2:]))
    #display(v)
    for col in v.columns[2:]: # <--- excluding Price and ID
        min_val = v[col].min()
        v.loc[:,col].fillna(min_val, inplace=True)

In [257]:
# for winsorization and standardization

def winsorize_nan(s, limits):
    return s.clip(lower=s.quantile(limits[0], interpolation='lower'), 
                  upper=s.quantile(1-limits[1], interpolation='higher'))

def winsorize_index(series):
    win_series = winsorize_nan(series,limits=(0.025,0.025))
    return win_series

def clean_src(dct,date):
    
    src_df = dct[date]

    ids = src_df.iloc[:,0:2]
    vals = src_df.iloc[:,2:]

    #winsorized
    win_container = {}
    for i in vals.columns:
        win_container[i] = winsorize_index(src_df[i])

    #standardized
    z_array = standard_scaler.fit_transform(pd.DataFrame(win_container))
    z_container = pd.DataFrame(z_array,index=src_df.index, columns=vals.columns)

    clean_src = pd.concat([ids,z_container],axis=1)

    return clean_src

rf_src = pd.DataFrame()
#rf_src = {}
for k,v in dct_src.items():
    #rf_src[k] = clean_src(dct_src,k)
    rf_src = pd.concat([rf_src,clean_src(dct_src,k)])

In [253]:
#PER PERIOD PRE PROCESSING
#standardize - DONE
#winsorize - DONE
#process NaN by replacing min() for the entire series - DONE
#input minimum number of available data per factor, if less than min then considered no data. - PARTIAL (threshold nan covers)

#turnover reduction function (e.g. per period)
#use quintiles to get highest magnitude per factor

In [265]:
rf_src.reset_index().info() #[pd.Timestamp('2025-05-31')]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   DATE           2225 non-null   datetime64[ns]
 1   stock          2225 non-null   object        
 2   price          2225 non-null   float64       
 3   hist_pb        2225 non-null   float64       
 4   hist_earn_yld  2225 non-null   float64       
 5   fwd_eps_bxo    2225 non-null   float64       
 6   fwd_dps        2225 non-null   float64       
 7   hist_eps       2225 non-null   float64       
 8   hist_sps       2225 non-null   float64       
 9   fwd_pe         2225 non-null   float64       
 10  fwd_eps        2225 non-null   float64       
 11  fwd_eps_STg    2225 non-null   float64       
 12  fwd_eps_LTg    2225 non-null   float64       
 13  fwd_dvy        2225 non-null   float64       
dtypes: datetime64[ns](1), float64(12), object(1)
memory usage: 243.5+ KB


In [261]:


# Value and growth factor definitions
value_factors = ['hist_pb', 'hist_earn_yld', 'fwd_dps', 'fwd_dvy']
growth_factors = ['hist_eps', 'hist_sps', 'fwd_pe', 'fwd_eps', 'fwd_eps_STg', 'fwd_eps_LTg']
all_factors = value_factors + growth_factors

# Load your data
df = rf_src.reset_index() #pd.read_csv('test_source.csv')

# Columns
df.rename(columns={'DATE':'date','ID':'stock'}, inplace=True)

# Ensure date column is datetime
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['stock', 'date'])

# Calculate next month's return for each stock
df['next_price'] = df.groupby('stock')['price'].shift(-1)
df['next_return'] = (df['next_price'] / df['price']) - 1

# Remove rows where next_return is NaN (usually last date per stock)
df = df.dropna(subset=['next_return'])

results = []

for date, date_df in df.groupby('date'):
    for stock, stock_df in date_df.groupby('stock'):
        # We just want the current observation to predict the next return
        # Only run RF if enough samples are available historically (say at least 12 for stability)
        hist_df = df[(df['stock'] == stock) & (df['date'] < date)]
        
        if len(hist_df) < 2:
            continue

        X = hist_df[all_factors]
        y = hist_df['next_return']
        
        if X.isnull().any().any() or y.isnull().any():
            continue  # Skip if any NA remains for features or target

        rf = RandomForestRegressor(n_estimators=100, random_state=42)
        rf.fit(X, y)
        importances = rf.feature_importances_

        # Identify important features (above mean importance)
        threshold = importances.mean() * 1.75
        important_features = [f for f, imp in zip(all_factors, importances) if imp > threshold]

        # Classify
        if important_features:
            is_value = all(f in value_factors for f in important_features)
            is_growth = all(f in growth_factors for f in important_features)
            if is_value:
                classification = 'Value'
            elif is_growth:
                classification = 'Growth'
            else:
                classification = 'Both'
        else:
            classification = 'Unclassified'  # or skip

        # For the current row (date, stock), get magnitudes for important features
        curr_row = stock_df.iloc[0]
        imp_feature_sum = curr_row[important_features].abs().sum() if important_features else np.nan

        results.append({
            'date': date,
            'stock': stock,
            'classification': classification,
            'important_features': important_features,
            'imp_feature_sum': imp_feature_sum
        })
        #print(results)

# Compile results
results_df = pd.DataFrame(results).dropna(subset=['imp_feature_sum'])

# Weighting within each group per date
final_rows = []
for (date, cls), group in results_df.groupby(['date', 'classification']):
    total = group['imp_feature_sum'].sum()
    if total > 0:
        group = group.copy()
        group['weight'] = group['imp_feature_sum'] / total
        for _, row in group.iterrows():
            final_rows.append(row)

final_df = pd.DataFrame(final_rows)

# Save or display
#final_df.to_csv('stock_classification_and_weights.csv', index=False)
print(final_df.head(20))


          date            stock classification        important_features  \
36  2020-08-31    AGI PM Equity          Value                 [fwd_dvy]   
39  2020-08-31    BDO PM Equity          Value                 [fwd_dvy]   
44  2020-08-31    EMI PM Equity          Value                 [fwd_dvy]   
98  2020-09-30    URC PM Equity           Both        [fwd_dvy, fwd_eps]   
67  2020-09-30   ACEN PM Equity         Growth         [fwd_pe, fwd_eps]   
68  2020-09-30    AEV PM Equity         Growth                [hist_eps]   
69  2020-09-30    AGI PM Equity         Growth                  [fwd_pe]   
72  2020-09-30    BDO PM Equity         Growth                [hist_sps]   
74  2020-09-30    BPI PM Equity         Growth                  [fwd_pe]   
78  2020-09-30   FGEN PM Equity         Growth                [hist_eps]   
80  2020-09-30  GTCAP PM Equity         Growth                [hist_eps]   
83  2020-09-30    JGS PM Equity         Growth                  [fwd_pe]   
91  2020-09-

In [262]:
final_df #.to_csv('test_results')

,date,stock,classification,important_features,imp_feature_sum,weight
36,2020-08-31,AGI PM Equity,Value,[fwd_dvy],0.249830,0.171834
39,2020-08-31,BDO PM Equity,Value,[fwd_dvy],0.539540,0.371098
44,2020-08-31,EMI PM Equity,Value,[fwd_dvy],0.664532,0.457068
98,2020-09-30,URC PM Equity,Both,"[fwd_dvy, fwd_eps]",0.282850,1.000000
67,2020-09-30,ACEN PM Equity,Growth,"[fwd_pe, fwd_eps]",0.755353,0.145565
...,...,...,...,...,...,...
2095,2025-04-30,RLC PM Equity,Value,[hist_pb],0.835652,0.098239
2098,2025-04-30,SECB PM Equity,Value,"[hist_earn_yld, fwd_dvy]",1.972950,0.231940
2102,2025-04-30,UBP PM Equity,Value,[fwd_dvy],0.255509,0.030038
2103,2025-04-30,URC PM Equity,Value,[fwd_dvy],0.304283,0.035772


In [264]:
print(df.info())
print(df.describe())


#df['fwd_dps'] = df['fwd_dps'].fillna(0)
#df['fwd_dvy'] = df['fwd_dvy'].fillna(0)

#df_clean =

df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2185 entries, 0 to 2188
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           2185 non-null   datetime64[ns]
 1   stock          2185 non-null   object        
 2   price          2185 non-null   float64       
 3   hist_pb        2185 non-null   float64       
 4   hist_earn_yld  2185 non-null   float64       
 5   fwd_eps_bxo    2185 non-null   float64       
 6   fwd_dps        2185 non-null   float64       
 7   hist_eps       2185 non-null   float64       
 8   hist_sps       2185 non-null   float64       
 9   fwd_pe         2185 non-null   float64       
 10  fwd_eps        2185 non-null   float64       
 11  fwd_eps_STg    2185 non-null   float64       
 12  fwd_eps_LTg    2185 non-null   float64       
 13  fwd_dvy        2185 non-null   float64       
 14  next_price     2185 non-null   float64       
 15  next_return    2185 n

date             0
stock            0
price            0
hist_pb          0
hist_earn_yld    0
fwd_eps_bxo      0
fwd_dps          0
hist_eps         0
hist_sps         0
fwd_pe           0
fwd_eps          0
fwd_eps_STg      0
fwd_eps_LTg      0
fwd_dvy          0
next_price       0
next_return      0
dtype: int64